# Jaw Keypoint Prediction

Set `CONDITION`, `CHECKPOINT`, and `INPUT_DIR` below, then run all cells.

Output: `predictions.csv` with `frame, tip_x, tip_y, line_x, line_y` in original 640×480 pixel coordinates.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import cv2
import pandas as pd

sys.path.insert(0, str(Path("utils.py").resolve().parent))
from utils import run_inference_on_image_dir, CONDITIONS

# ── Configuration ──────────────────────────────────────────────────────────
CONDITION = "IRt_BiPoles"  # "IRt_BiPoles" | "IRt_TeLC" | "PCRt_BiPoles"
CHECKPOINT = "../Training/checkpoints/best_model.pt"
INPUT_DIR = "/mnt/c/Users/wanglab/Desktop/Tip+Base/IRt_BiPoles/images"
OUTPUT_CSV = "predictions.csv"

assert CONDITION in CONDITIONS, f"Choose one of {CONDITIONS}"

In [ ]:
df = run_inference_on_image_dir(
    checkpoint_path=CHECKPOINT,
    input_dir=INPUT_DIR,
    condition=CONDITION,
    output_csv=OUTPUT_CSV,
)
df.head()

In [ ]:
# Overlay predictions on a few sample frames
img_dir = Path(INPUT_DIR)
sample_frames = df["frame"].dropna().astype(int).iloc[:: max(1, len(df) // 5)][:5]

fig, axes = plt.subplots(1, len(sample_frames), figsize=(4 * len(sample_frames), 4))
if len(sample_frames) == 1:
    axes = [axes]

for ax, fr in zip(axes, sample_frames):
    path = img_dir / f"{fr:07d}.png"
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    row = df.loc[df["frame"] == fr].iloc[0]
    ax.imshow(img)
    ax.scatter(row["tip_x"], row["tip_y"], c="lime", s=40, label="tip")
    ax.scatter(row["line_x"], row["line_y"], c="red", s=40, label="line")
    ax.set_title(f"frame {fr}")
    ax.axis("off")

axes[0].legend(loc="upper right")
plt.tight_layout()
plt.show()